In [ ]:
import pickle
import csv
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

In [ ]:
with open('data3.pickle', 'rb') as f:
    data = pickle.load(f, encoding='latin1')

x_train = data['x_train'].astype(np.float32)
y_train = data['y_train'].astype(np.int64)
x_val   = data['x_validation'].astype(np.float32)
y_val   = data['y_validation'].astype(np.int64)
x_test  = data['x_test'].astype(np.float32)
y_test  = data['y_test'].astype(np.int64)

def load_label_names(file):
    names = []
    with open(file, 'r') as f:
        reader = csv.reader(f)
        for row in reader:
            names.append(row[1])
    return names[1:]

label_names = load_label_names('label_names.csv')
NUM_CLASSES = len(label_names)

print(f'Train      : {x_train.shape}')
print(f'Validation : {x_val.shape}')
print(f'Test       : {x_test.shape}')
print(f'Classes    : {NUM_CLASSES}')

In [ ]:
BATCH_SIZE = 128

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_val), torch.from_numpy(y_val)),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test)),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True
)

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=128):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)       # (B, D, H/P, W/P)
        x = x.flatten(2)       # (B, D, N)
        x = x.transpose(1, 2)  # (B, N, D)
        return x


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, attn_drop=0.0, proj_drop=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=attn_drop, batch_first=True)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x, return_weights=False):
        x_norm = self.norm(x)
        attn_out, attn_weights = self.attn(x_norm, x_norm, x_norm)
        attn_out = self.proj_drop(attn_out)
        out = x + attn_out
        if return_weights:
            return out, attn_weights
        return out


class MLP(nn.Module):
    def __init__(self, embed_dim, mlp_ratio=4, dropout=0.1):
        super().__init__()
        hidden_dim = int(embed_dim * mlp_ratio)
        self.norm  = nn.LayerNorm(embed_dim)
        self.fc1   = nn.Linear(embed_dim, hidden_dim)
        self.act   = nn.GELU()
        self.drop1 = nn.Dropout(dropout)
        self.fc2   = nn.Linear(hidden_dim, embed_dim)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x):
        h = self.norm(x)
        h = self.drop2(self.fc2(self.drop1(self.act(self.fc1(h)))))
        return x + h


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4, attn_drop=0.0, proj_drop=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, attn_drop, proj_drop)
        self.mlp  = MLP(embed_dim, mlp_ratio, proj_drop)

    def forward(self, x, return_weights=False):
        if return_weights:
            x, w = self.attn(x, return_weights=True)
            return self.mlp(x), w
        return self.mlp(self.attn(x))


class VisionTransformer(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3,
                 num_classes=43, embed_dim=128, depth=6, num_heads=8,
                 mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches      = self.patch_embed.num_patches
        self.cls_token   = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed   = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop    = nn.Dropout(dropout)
        self.blocks      = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, proj_drop=dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.zeros_(m.bias)
            nn.init.ones_(m.weight)

    def forward(self, x, return_attn=False):
        B  = x.size(0)
        x  = self.patch_embed(x)
        x  = torch.cat([self.cls_token.expand(B, -1, -1), x], dim=1)
        x  = self.pos_drop(x + self.pos_embed)
        attn_weights_all = []
        for i, block in enumerate(self.blocks):
            if return_attn and i == len(self.blocks) - 1:
                x, w = block(x, return_weights=True)
                attn_weights_all.append(w)
            else:
                x = block(x)
        x      = self.norm(x)
        logits = self.head(x[:, 0])
        if return_attn:
            return logits, attn_weights_all
        return logits

In [ ]:
if DEVICE.type == 'cuda':
    cfg = dict(patch_size=4, embed_dim=128, depth=6, num_heads=8)
    print('GPU → configuration complète (patch=4, dim=128, depth=6)')
else:
    cfg = dict(patch_size=8, embed_dim=64, depth=4, num_heads=4)
    print('CPU → configuration allégée (patch=8, dim=64, depth=4)')

model = VisionTransformer(
    img_size=32, in_channels=3, num_classes=NUM_CLASSES,
    mlp_ratio=4, dropout=0.1, **cfg
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Paramètres entraînables : {total_params:,}')

dummy = torch.randn(4, 3, 32, 32).to(DEVICE)
print(f'Test forward : {dummy.shape} → {model(dummy).shape}')

In [ ]:
NUM_EPOCHS    = 30
LEARNING_RATE = 3e-4
WARMUP_EPOCHS = 5
MIN_LR        = 1e-6

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05)

def get_lr(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, NUM_EPOCHS - WARMUP_EPOCHS)
    return max(MIN_LR / LEARNING_RATE, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if train:
                optimizer.zero_grad()
            logits = model(xb)
            loss   = criterion(logits, yb)
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item() * xb.size(0)
            correct    += (logits.argmax(1) == yb).sum().item()
            total      += xb.size(0)
    return total_loss / total, correct / total


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader,   train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), 'best_vit_model.pth')

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} '
          f'| train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} '
          f'| val_loss={va_loss:.4f} val_acc={va_acc:.4f}')

print(f'\nMeilleure val_acc : {best_val_acc:.4f}')

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs, history['train_loss'], label='Train')
ax1.plot(epochs, history['val_loss'],   label='Validation')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, history['train_acc'], label='Train')
ax2.plot(epochs, history['val_acc'],   label='Validation')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('vit_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
model.load_state_dict(torch.load('best_vit_model.pth', map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print(f'Accuracy test : {accuracy_score(all_labels, all_preds):.4f}')
print(classification_report(
    all_labels, all_preds,
    target_names=[f'{i}: {n[:20]}' for i, n in enumerate(label_names)],
    digits=3
))

In [ ]:
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm_norm, annot=False, cmap='Blues',
            xticklabels=range(NUM_CLASSES),
            yticklabels=range(NUM_CLASSES),
            vmin=0, vmax=1, ax=ax)
ax.set_xlabel('Prédiction')
ax.set_ylabel('Vérité terrain')
ax.set_title('Matrice de confusion normalisée — ViT')
plt.tight_layout()
plt.savefig('vit_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
with open('mean_image_rgb.pickle', 'rb') as f:
    mean_rgb = pickle.load(f, encoding='latin1')['mean_image_rgb']
with open('std_rgb.pickle', 'rb') as f:
    std_rgb = pickle.load(f, encoding='latin1')['std_rgb']

def denormalize(img):
    img = np.clip((img * std_rgb + mean_rgb) * 255.0, 0, 255).astype(np.uint8)
    return img.transpose(1, 2, 0)

def get_attention_map(image_tensor):
    model.eval()
    with torch.no_grad():
        _, attn_list = model(image_tensor.unsqueeze(0).to(DEVICE), return_attn=True)
    attn     = attn_list[-1][0]  # (N+1, N+1)
    cls_attn = attn[0, 1:]       # (N,)
    side     = int(math.sqrt(cls_attn.shape[0]))
    return cls_attn.reshape(side, side).cpu().numpy()

def resize_map(attn_map, size=32):
    t = torch.from_numpy(attn_map).unsqueeze(0).unsqueeze(0).float()
    t = torch.nn.functional.interpolate(t, size=(size, size), mode='bilinear', align_corners=False)
    return t.squeeze().numpy()


rng  = np.random.default_rng(7)
idxs = rng.choice(len(x_test), size=6, replace=False)

fig, axes = plt.subplots(6, 3, figsize=(9, 18))
for row, idx in enumerate(idxs):
    img_np   = x_test[idx]
    img_vis  = denormalize(img_np)
    attn_map = resize_map(get_attention_map(torch.from_numpy(img_np)))
    color    = 'green' if y_test[idx] == all_preds[idx] else 'red'

    axes[row, 0].imshow(img_vis)
    axes[row, 0].set_title(f'Vrai:{y_test[idx]} Préd:{all_preds[idx]}', color=color, fontsize=8)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(attn_map, cmap='hot')
    axes[row, 1].set_title('Attention map', fontsize=8)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(img_vis)
    axes[row, 2].imshow(attn_map, cmap='hot', alpha=0.5)
    axes[row, 2].set_title('Superposition', fontsize=8)
    axes[row, 2].axis('off')

fig.suptitle('Cartes d\'attention — ViT', fontsize=13)
plt.tight_layout()
plt.savefig('vit_attention_maps.png', dpi=150, bbox_inches='tight')
plt.show()
